<center>
    <p style="text-align:center">
        <img alt="phoenix logo" src="https://storage.googleapis.com/arize-phoenix-assets/assets/phoenix-logo-light.svg" width="200"/>
        <br>
        <a href="https://arize.com/docs/phoenix/">Docs</a>
        |
        <a href="https://github.com/Arize-ai/phoenix">GitHub</a>
        |
        <a href="https://arize-ai.slack.com/join/shared_invite/zt-2w57bhem8-hq24MB6u7yE_ZF_ilOYSBw#/shared-invite/email">Community</a>
    </p>
</center>
<h1 align="center">Creating a Custom LLM Evaluator with a Benchmark Dataset</h1>

A good evaluator measures what you actually care about — and the only way to know it does is to test it against examples you've judged yourself. This notebook builds a custom LLM-as-a-Judge evaluator the same way the pre-built ones are built, in three steps:

1. **Build a benchmark dataset** — human-labeled examples that capture your definition of "good."
2. **Write the judge prompt** — the LLM-as-a-Judge template that encodes that definition.
3. **Validate before you trust it** — measure how well the judge agrees with your labels, and iterate until it does.

See the full write-up in the [cookbook](https://arize.com/docs/phoenix/cookbook/evaluation/creating-a-custom-llm-evaluator-with-a-benchmark-dataset).

## Setup

Install dependencies, connect to Phoenix, and auto-instrument OpenAI so every call is traced to the `receipt-classification` project.

In [ ]:
%pip install -q arize-phoenix-client arize-phoenix-evals arize-phoenix-otel openinference-instrumentation-openai openai pandas

In [ ]:
import os
from getpass import getpass

from phoenix.otel import register

# Prompt for any keys not already set in the environment (don't clobber real values).
if not os.environ.get("PHOENIX_API_KEY"):
    os.environ["PHOENIX_API_KEY"] = getpass("Enter your Phoenix API key: ")
if not os.environ.get("OPENAI_API_KEY"):
    os.environ["OPENAI_API_KEY"] = getpass("Enter your OpenAI API key: ")
# For Phoenix Cloud, also point at your space's collector endpoint, e.g.:
# os.environ["PHOENIX_COLLECTOR_ENDPOINT"] = "https://app.phoenix.arize.com/s/your-space"

tracer_provider = register(
    project_name="receipt-classification",
    auto_instrument=True,  # instruments installed libraries, including OpenAI
)

## Generate receipt extraction traces

We ask GPT-5.5 to turn receipt images into short expense reports. The calls are auto-traced to Phoenix, where you'd normally annotate them. The sample images are public receipts from [Wikimedia Commons](https://commons.wikimedia.org/wiki/Category:Receipts).

In [ ]:
from openai import OpenAI

client = OpenAI()


def extract_receipt_data(image_url):
    response = client.chat.completions.create(
        model="gpt-5.5",
        max_tokens=500,
        messages=[
            {
                "role": "user",
                "content": [
                    {
                        "type": "text",
                        "text": "Analyze this receipt and return a brief summary for an expense report. Only include category of expense, total cost, and summary of items",
                    },
                    {"type": "image_url", "image_url": {"url": image_url}},
                ],
            }
        ],
    )
    return response.choices[0].message.content


urls = [
    "https://upload.wikimedia.org/wikipedia/commons/2/25/Receipt.jpg",
    "https://upload.wikimedia.org/wikipedia/commons/d/df/Save_Mart_recipt_2010-10-23.jpg",
    "https://upload.wikimedia.org/wikipedia/commons/9/9e/Restaurant_Bill_1_2013-07-08.jpg",
]
for url in urls:
    print(extract_receipt_data(url)[:120], "...\n")

## Build a benchmark dataset

In practice you'd annotate the traces above in the Phoenix UI and pull them into a dataset (see the [cookbook](https://arize.com/docs/phoenix/cookbook/evaluation/creating-a-custom-llm-evaluator-with-a-benchmark-dataset)). So the notebook runs end-to-end, we use a small **pre-labeled** set here.

Note: `create_dataset` serializes the DataFrame as CSV, so a nested column like `attributes.llm.output_messages` is stored as a string. We flatten the model output to a plain-text column up front so the evaluator can read it.

In [ ]:
import pandas as pd
from phoenix.client import Client

# Stand-in for annotated traces: (input, model expense-report output, human label).
rows = [
    (
        "receipt_office.jpg",
        "Category: Office Supplies\nTotal: $47.32\nItems: paper, stapler, pens",
        "accurate",
    ),
    (
        "receipt_dinner.jpg",
        "Category: Meals\nTotal: $86.10\nItems: 2 entrees, 1 appetizer, 2 drinks",
        "accurate",
    ),
    ("receipt_vague.jpg", "Category: Misc\nTotal: $15.00\nItems: stuff", "almost accurate"),
    ("receipt_nototal.jpg", "Category: Travel\nSummary: round-trip train ticket", "inaccurate"),
    ("receipt_offtask.jpg", "The image shows a sunset over the ocean.", "inaccurate"),
]
full_df = pd.DataFrame(
    {
        "attributes.input.value": [r[0] for r in rows],
        "attributes.llm.output_messages": [[{"message.content": r[1]}] for r in rows],
        "result.label": [r[2] for r in rows],
    }
)


def first_message_content(messages):
    # get_spans_dataframe returns output_messages as a list of message dicts;
    # guard against empty rows (and the occasional already-stringified value).
    if isinstance(messages, list) and messages:
        return messages[0].get("message.content", "")
    if isinstance(messages, str):
        return messages
    return ""


full_df["output_text"] = full_df["attributes.llm.output_messages"].apply(first_message_content)

px_client = Client()
dataset = px_client.datasets.create_dataset(
    name="annotated-receipts",
    dataframe=full_df,
    input_keys=["attributes.input.value"],
    output_keys=["output_text"],
    metadata_keys=["result.label"],
)

## Write the judge prompt and run an experiment

The task runs the judge on each example; the evaluator scores agreement with the human label. We start with a loose template to establish a baseline.

In [ ]:
from phoenix.client.experiments import run_experiment
from phoenix.evals import LLM, ClassificationEvaluator

choices = ["accurate", "almost accurate", "inaccurate"]
llm = LLM(provider="openai", model="gpt-5.5")

initial_template = """You are an evaluator assessing the quality of a model-generated expense report.
MODEL OUTPUT (Expense Report): {output}
Assign one of the following labels. Only include the label:
- "accurate" - includes expense category, total cost, and item summary; all information looks reasonable
- "almost accurate" - mostly correct but with small issues (e.g. missing one element or vague category)
- "inaccurate" - substantially wrong or missing critical information
"""


def evaluate_response(output, metadata):
    return 1 if metadata["result.label"] == output else 0


def make_task(template):
    evaluator = ClassificationEvaluator(
        name="receipt_accuracy", prompt_template=template, llm=llm, choices=choices
    )

    def task_function(input, reference):
        output = reference["output_text"]
        return evaluator.evaluate(eval_input={"output": output})[0].label

    return task_function


initial_experiment = run_experiment(
    dataset=dataset,
    task=make_task(initial_template),
    evaluators=[evaluate_response],
    experiment_name="initial template",
)

## Iterate on the judge prompt

Tighten the rules to close the gaps the first run exposed, then re-run to compare agreement.

In [ ]:
improved_template = """You are an evaluator assessing the quality of a model-generated expense report.
MODEL OUTPUT (Expense Report): {output}
Assign one of the following labels. Only include the label:
- "accurate" - total price, itemized list, AND expense category are all present and reasonable. All three must be present.
- "almost accurate" - mostly correct but with a small issue (e.g. category too vague or one element missing).
- "inaccurate" - substantially wrong or missing critical information, e.g. missing the total price entirely.
"""

improved_experiment = run_experiment(
    dataset=dataset,
    task=make_task(improved_template),
    evaluators=[evaluate_response],
    experiment_name="improved template",
)

## Validate before you trust it

Each experiment reports how often the judge's label matched the human label — that agreement score is your validation signal. Compare the two runs in Phoenix: the improved template should agree with your benchmark more often. Don't trust the judge in production until it aligns with your benchmark; when it falls short, inspect the disagreements and either tighten the prompt or refine your labels.